In [3]:
import pandas as pd 

sales_train = pd.read_csv('D:\Datathon\data\sales.csv')

df = sales_train.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

ts = df.set_index('Date')['Revenue']

<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14832\73792621.py:3: SyntaxWarning: invalid escape sequence '\D'
  sales_train = pd.read_csv('D:\Datathon\data\sales.csv')


In [4]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(ts)
print(result[1])  # p-value

0.10949851106912595


In [5]:
ts_diff = ts.diff().dropna()

In [6]:
result = adfuller(ts_diff)
print(result[1]) #D = 1

0.0


In [7]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    ts,
    order=(1,1,1),
    seasonal_order=(1,1,1,7) 
    # s = 365, chu kỳ theo năm
    # D = 1, loại bỏ pattern lặp lại mỗi 365 ngày
    # P = 1, hôm nay phụ thuộc vào cùng ngày tuần trước, p=1: dùng lag của 1 ngày trước
    # Q = 1, error tại t-365
)

result = model.fit()
print(result.summary())

c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


                                     SARIMAX Results                                     
Dep. Variable:                           Revenue   No. Observations:                 3833
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 7)   Log Likelihood              -59592.333
Date:                           Tue, 28 Apr 2026   AIC                         119194.665
Time:                                   19:45:45   BIC                         119225.912
Sample:                               07-04-2012   HQIC                        119205.766
                                    - 12-31-2022                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.9093      0.052    -17.574      0.000      -1.011      -0.808
ma.L1          0.8584      0.063     13.570

# parameters tuning

In [10]:
import itertools

p = d = q = range(0, 2)
P = D = Q = range(0, 2)
s = 7

best_aic = float("inf")
best_params = None

for param in itertools.product(p, d, q):
    for seasonal in itertools.product(P, D, Q):
        try:
            model = SARIMAX(ts,
                            order=param,
                            seasonal_order=(*seasonal, s))
            res = model.fit(disp=False)
            
            if res.aic < best_aic:
                best_aic = res.aic
                best_params = (param, seasonal)
        except:
            continue

print(best_params)

c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Administrator\AppDa

((1, 1, 1), (1, 0, 1))


In [13]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    ts,  
    order=best_params[0],
    seasonal_order=(*best_params[1], 7)
)

result = model.fit()

c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [14]:
import pandas as pd 

sample_submission = pd.read_csv(r'D:\Datathon\data\sample_submission.csv')
n_test = len(sample_submission)
n_test

548

In [ ]:
forecast = result.forecast(steps=n_test)

submission = sample_submission.copy()

submission['Revenue'] = forecast.values

submission.to_csv('sub.csv', index=False)

print(submission.head())

         Date       Revenue        COGS
0  2023-01-01  2.377704e+06  2518885.15
1  2023-01-02  2.445975e+06  1136463.00
2  2023-01-03  2.296148e+06   822721.12
3  2023-01-04  2.252641e+06   914554.18
4  2023-01-05  2.216520e+06   984390.24
